# Basic

In [ ]:
%load_ext autoreload
%autoreload all

In [ ]:
import polars as pl
import pickle
import numpy as np
import tqdm
import os
import glob
import multiprocessing as mp

import src.graph_tokenizer_gd_tree_dev.config as config
import src.graph_tokenizer_gd_tree_dev.tokenizer as tokenizer
import src.graph_tokenizer_gd_tree_dev.eval as eval

In [ ]:
def to_type_dict(file_list):
    return {os.path.splitext(os.path.basename(f))[0]: f for f in file_list}

In [ ]:
gd_tree_list = glob.glob(config.CandidateLists().path_greedy_tree + '/*.parquet')
baseline_list = glob.glob(config.CandidateLists().baseline_path + '/*.parquet')

all_candidates = {
    "greedy_tree_margin": to_type_dict(gd_tree_list),
    "baseline": to_type_dict(baseline_list),
}
for category, file_list in all_candidates.items():
    for file_type, file in file_list.items():
        print(f"Category: {category}, File Type: {file_type}, File: {file}")

# compute metrics

In [ ]:
with open(config.ProcessedGraph().combined_subgraphs, "rb") as f:
    combined_subgraphs = pickle.load(f)

with open(config.ProcessedGraph().id_to_label, "rb") as f:
    id_to_label = pickle.load(f)

df_mapped = pl.read_parquet(f"{config.BasicConfig().mapped_path}")
mapped_ids = df_mapped["id"].unique().to_list()
Ks = config.TokenizerParam().Ks
rnd_iters = config.TokenizerParam().rnd_iters

In [ ]:
results = []
candidate_col = "token"
tasks = []
for category, file_list in all_candidates.items():
    for file_type, file in file_list.items():
        if file_type == "k_random_all_samples":
            continue
        df = pl.read_parquet(file)
        for k in Ks:
            candidates = df.head(k)[candidate_col].to_list()
            tasks.append(((category, file_type, k), candidates))

# Built once and shared across every task instead of being rebuilt per (category, file_type, k):
# A/node_to_idx is the semantic-coverage transition matrix, adj is the out-adjacency used by
# context-tree building. Neither depends on the candidate set T, only on the fixed graph.
A, node_to_idx = tokenizer.build_coverage_transition(combined_subgraphs)
adj = tokenizer.build_out_adjacency(combined_subgraphs)

n_workers = os.cpu_count()
with mp.Pool(n_workers, initializer=tokenizer._init_coverage_worker,
             initargs=(A, node_to_idx, adj, mapped_ids, config.TokenizerParam.max_dist_candidate, id_to_label)) as pool:
    task_results = pool.imap_unordered(tokenizer._worker_coverage_score, tasks, chunksize=4)
    for (category, file_type, k), metrics in tqdm.tqdm(task_results, total=len(tasks)):
        results.append({
            "category": category,
            "file_type": file_type,
            "k": k,
            **metrics,
        })

results_df = pl.DataFrame(results)
results_df.write_parquet(config.Results().perf_greedy_tree_path)

In [ ]:
results_k = []
candidate_col = "token"
category, file_type = "baseline", "k_random_all_samples"
df = pl.read_parquet(config.CandidateLists().k_random_all_samples)
tasks = [
    ((k, it), df.filter((pl.col("iter") == it) & (pl.col("k") == k))[candidate_col].to_list())
    for k in Ks
    for it in rnd_iters
]
n_workers = os.cpu_count()
with mp.Pool(n_workers, initializer=tokenizer._init_coverage_worker,
             initargs=(A, node_to_idx, adj, mapped_ids, config.TokenizerParam.max_dist_candidate, id_to_label)) as pool:
    task_results = pool.imap_unordered(tokenizer._worker_coverage_score, tasks, chunksize=4)
    for (k, it), metrics in tqdm.tqdm(task_results, total=len(tasks)):
        results_k.append({
            "category": category,
            "file_type": file_type,
            "k": k,
            "iter": it,
            **metrics,
        })

results_k_df = pl.DataFrame(results_k)
results_k_df.write_parquet(config.Results().perf_baseline_path)


# plots for comparison

In [ ]:
import matplotlib.pyplot as plt

metric_cols = ["semantic_coverage", "conciseness", "distance_score", "uniqueness_entropy",
               "unk_branche_rate", "uncovered_rate", "tree_complexity", "exact_rate", "unique_rate"]

# perf_df: greedy_tree_margin (6 lam values) + the 3 single-ranking baseline heuristics, straight
# from disk -- robust to kernel restarts, and this is the persisted, confirmed-complete output of
# the (category, file_type, k) sweep above.
perf_df = pl.read_parquet(config.Results().perf_greedy_tree_path).rename({"unk_rate": "unk_branche_rate"}).sort(["category", "file_type", "k"])
# perf_df_k: k_random_all_samples, averaged over `iter` per k so it's comparable to the
# single-ranking file_types above.
perf_df_k = (pl.read_parquet(config.Results().perf_baseline_path)
             .rename({"unk_rate": "unk_branche_rate"})
             .sort(["category", "file_type", "k"])
             .group_by(["category", "file_type", "k"])
             .agg(pl.col(metric_cols).mean()))
perf_df_all = pl.concat([perf_df, perf_df_k], how="diagonal_relaxed").sort(["category", "file_type", "k"]).filter(pl.col("file_type")!= "eigenvector_centrality")  # eigenvector_centrality is a degenerate baseline that always returns the same candidate set for all k, so it doesn't make sense to include it in the plots.

series_keys = perf_df_all.select(["category", "file_type"]).unique().sort(["category", "file_type"]).rows()

# Hue-varied ordinal ramp: red -> yellow -> green across all series, fixed sorted order.
cmap = plt.get_cmap("RdYlGn")
colors = {key: cmap(i / max(len(series_keys) - 1, 1)) for i, key in enumerate(series_keys)}
# baseline = dotted ("pointillé"), greedy_tree_margin = solid -- a second, category-level
# encoding on top of color so the two arms are distinguishable even without reading the legend.
linestyles = {key: (":" if key[0] == "baseline" else "-") for key in series_keys}

fig, axes = plt.subplots(2, 5, figsize=(25, 9))
axes = axes.flatten()

for ax, metric in zip(axes, metric_cols):
    for category, file_type in series_keys:
        sub = perf_df_all.filter((pl.col("category") == category) & (pl.col("file_type") == file_type))
        ax.plot(sub["k"], sub[metric], color=colors[(category, file_type)], linestyle=linestyles[(category, file_type)],
                 linewidth=2, marker="o", markersize=3)
    ax.set_title(metric.replace("_", " "))
    ax.set_xlabel("k")
    ax.grid(alpha=0.3)

axes[-1].axis("off")  # 10th slot unused by metrics -- holds the shared legend instead
handles = [
    plt.Line2D([0], [0], color=colors[key], linestyle=linestyles[key], lw=2, marker="o", markersize=4,
               label=f"{key[0]} / {key[1]}")
    for key in series_keys
]
axes[-1].legend(handles=handles, loc="center", fontsize=9, title="category / file_type", frameon=False)

fig.suptitle("Candidate set performance vs k", fontsize=14)
fig.tight_layout()
plt.show()

# choose certain plot only 

In [ ]:
metric_cols = ["semantic_coverage", "unique_rate", "distance_score", "conciseness", "uncovered_rate", "tree_complexity"]
all_cols = ["category", "file_type", "k"] + metric_cols
# perf_df_chosen  = perf_df_all.select(all_cols).filter((pl.col("category") == "greedy_tree_margin"))
perf_df_chosen  = perf_df_all.select(all_cols).filter((pl.col("category") == "baseline") & (pl.col("file_type") != "k_random_all_samples"))


series_keys = perf_df_chosen.select(["category", "file_type"]).unique().sort(["category", "file_type"]).rows()

# Hue-varied ordinal ramp: red -> yellow -> green across all series, fixed sorted order.
cmap = plt.get_cmap("RdYlGn")
colors = {key: cmap(i / max(len(series_keys) - 1, 1)) for i, key in enumerate(series_keys)}
# baseline = dotted ("pointillé"), greedy_tree_margin = solid -- a second, category-level
# encoding on top of color so the two arms are distinguishable even without reading the legend.
linestyles = {key: ("-" if key[0] == "baseline" else "-") for key in series_keys}

fig, axes = plt.subplots(2, 3, figsize=(25, 9))
axes = axes.flatten()

for ax, metric in zip(axes, metric_cols):
    for category, file_type in series_keys:
        sub = perf_df_chosen.filter((pl.col("category") == category) & (pl.col("file_type") == file_type))
        ax.plot(sub["k"], sub[metric], color=colors[(category, file_type)], linestyle=linestyles[(category, file_type)],
                 linewidth=2, marker="o", markersize=3)
    ax.set_title(metric.replace("_", " "))
    ax.set_xlabel("k")
    ax.grid(alpha=0.3)

# All 6 axes are used by the 6 metrics -- unlike the 9-metric grid above, there's no spare
# slot to steal for the legend, so it goes on the figure itself instead.
handles = [
    plt.Line2D([0], [0], color=colors[key], linestyle=linestyles[key], lw=2, marker="o", markersize=4,
               label=f"{key[0]} / {key[1]}")
    for key in series_keys
]
fig.legend(handles=handles, loc="lower center", ncol=len(series_keys), fontsize=9,
           title="category / file_type", frameon=False)

fig.suptitle("Candidate set performance vs k", fontsize=14)
fig.tight_layout(rect=[0, 0.08, 1, 1])  # leave room at the bottom for the legend
plt.show()

In [ ]:
metric_cols = ["semantic_coverage", "unique_rate", "distance_score", "conciseness", "uncovered_rate", "tree_complexity"]
all_cols = ["category", "file_type", "k"] + metric_cols
perf_df_chosen  = perf_df_all.select(all_cols).filter((pl.col("category") == "greedy_tree_margin") & (~pl.col("file_type").str.contains("0.0")))
# perf_df_chosen  = perf_df_all.select(all_cols).filter((pl.col("category") == "baseline") & (pl.col("file_type") != "k_random_all_samples"))


series_keys = perf_df_chosen.select(["category", "file_type"]).unique().sort(["category", "file_type"]).rows()

# Hue-varied ordinal ramp: red -> yellow -> green across all series, fixed sorted order.
cmap = plt.get_cmap("RdYlGn")
colors = {key: cmap(i / max(len(series_keys) - 1, 1)) for i, key in enumerate(series_keys)}
# baseline = dotted ("pointillé"), greedy_tree_margin = solid -- a second, category-level
# encoding on top of color so the two arms are distinguishable even without reading the legend.
linestyles = {key: ("-" if key[0] == "baseline" else "-") for key in series_keys}

fig, axes = plt.subplots(2, 3, figsize=(25, 9))
axes = axes.flatten()

for ax, metric in zip(axes, metric_cols):
    for category, file_type in series_keys:
        sub = perf_df_chosen.filter((pl.col("category") == category) & (pl.col("file_type") == file_type))
        ax.plot(sub["k"], sub[metric], color=colors[(category, file_type)], linestyle=linestyles[(category, file_type)],
                 linewidth=2, marker="o", markersize=3)
    ax.set_title(metric.replace("_", " "))
    ax.set_xlabel("k")
    ax.grid(alpha=0.3)

# All 6 axes are used by the 6 metrics -- unlike the 9-metric grid above, there's no spare
# slot to steal for the legend, so it goes on the figure itself instead.
handles = [
    plt.Line2D([0], [0], color=colors[key], linestyle=linestyles[key], lw=2, marker="o", markersize=4,
               label=f"{key[0]} / {key[1]}")
    for key in series_keys
]
fig.legend(handles=handles, loc="lower center", ncol=len(series_keys), fontsize=9,
           title="category / file_type", frameon=False)

fig.suptitle("Candidate set performance vs k", fontsize=14)
fig.tight_layout(rect=[0, 0.08, 1, 1])  # leave room at the bottom for the legend
plt.show()

In [ ]:
metric_cols = ["semantic_coverage", "unique_rate", "distance_score", "conciseness", "uncovered_rate", "tree_complexity"]
all_cols = ["category", "file_type", "k"] + metric_cols
perf_df_chosen  = perf_df_all.select(all_cols).filter((pl.col("file_type").str.contains("0.8") | (pl.col("file_type")=="personalized_pagerank")) | (pl.col("file_type")=="discrete_set_cover"))
# perf_df_chosen  = perf_df_all.select(all_cols).filter((pl.col("category") == "baseline") & (pl.col("file_type") != "k_random_all_samples"))


series_keys = perf_df_chosen.select(["category", "file_type"]).unique().sort(["category", "file_type"]).rows()

# Hue-varied ordinal ramp: red -> yellow -> green across all series, fixed sorted order.
cmap = plt.get_cmap("RdYlGn")
colors = {key: cmap(i / max(len(series_keys) - 1, 1)) for i, key in enumerate(series_keys)}
colors[("baseline", "personalized_pagerank")] = "#F97316"  # solid blue, replaces the washed-out RdYlGn midpoint yellow

# baseline = dotted ("pointillé"), greedy_tree_margin = solid -- a second, category-level
# encoding on top of color so the two arms are distinguishable even without reading the legend.
linestyles = {key: ("-" if key[0] == "baseline" else "-") for key in series_keys}

fig, axes = plt.subplots(2, 3, figsize=(25, 9))
axes = axes.flatten()

for ax, metric in zip(axes, metric_cols):
    for category, file_type in series_keys:
        sub = perf_df_chosen.filter((pl.col("category") == category) & (pl.col("file_type") == file_type))
        ax.plot(sub["k"], sub[metric], color=colors[(category, file_type)], linestyle=linestyles[(category, file_type)],
                 linewidth=2, marker="o", markersize=3)
    ax.set_title(metric.replace("_", " "))
    ax.set_xlabel("k")
    ax.grid(alpha=0.3)

# All 6 axes are used by the 6 metrics -- unlike the 9-metric grid above, there's no spare
# slot to steal for the legend, so it goes on the figure itself instead.
handles = [
    plt.Line2D([0], [0], color=colors[key], linestyle=linestyles[key], lw=2, marker="o", markersize=4,
               label=f"{key[0]} / {key[1]}")
    for key in series_keys
]
fig.legend(handles=handles, loc="lower center", ncol=len(series_keys), fontsize=9,
           title="category / file_type", frameon=False)

fig.suptitle("Candidate set performance vs k", fontsize=14)
fig.tight_layout(rect=[0, 0.08, 1, 1])  # leave room at the bottom for the legend
plt.show()